<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Taller 1 · De Excel a Colab
**Laboratorio Excel + Colab** — Tu libro de Excel se convierte en un informe interactivo · ⏱ 45 min en Colab

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/taller_1_excel_a_colab.ipynb) &nbsp; [Ver el taller completo en el tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#taller-1)

Este cuaderno es una parte del taller. Los pasos en Excel están en el tablero.

**Cómo usarlo:** *Archivo › Guardar una copia en Drive* → *Entorno de ejecución › Ejecutar todas*. Las celdas con listas desplegables se pueden cambiar y volver a ejecutar con ▶. No necesitas programar.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

## Paso 3 · ¿De dónde viene tu libro?
Elige la fuente en la lista y ejecuta la celda con ▶. Si eliges *Subir desde mi computador*, aparece un botón **Elegir archivos**.

In [ ]:
# @title Leer el libro de Excel
fuente = "Copia del curso (GitHub)"  # @param ["Copia del curso (GitHub)", "Subir desde mi computador", "Desde mi Google Drive", "Copia ya subida al panel de Colab"]
ruta_en_drive = "Laboratorio_Andina/1_datos/Andina_listo_para_dinamicas.xlsx"  # @param {type:"string"}
nombre_en_panel = "Andina_listo_para_dinamicas.xlsx"  # @param {type:"string"}

if fuente == "Subir desde mi computador" and EN_COLAB:
    from google.colab import files
    archivo = list(files.upload())[0]
elif fuente == "Desde mi Google Drive" and EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    archivo = "/content/drive/MyDrive/" + ruta_en_drive
elif fuente == "Copia ya subida al panel de Colab":
    archivo = nombre_en_panel
else:
    archivo = ruta("Andina_listo_para_dinamicas.xlsx")

libro = pd.read_excel(archivo, sheet_name="Ventas")
libro["fecha"] = pd.to_datetime(libro["fecha"], errors="coerce")
display(Markdown(f"Leí **{miles(len(libro))} filas** y **{libro.shape[1]} columnas** de la hoja Ventas."))
display(Markdown(f"Primer cliente del libro: **{libro['cliente'].iloc[0]}**. Si lo cambiaste en Excel, aquí debe aparecer tu cambio."))

## Paso 4 · Diagnóstico automático
Antes de graficar se revisa. Es el hábito de la clave 12: comprobar antes de concluir.

In [ ]:
# @title Revisar el libro
esperadas = ["fecha", "anio", "mes", "ciudad", "vendedor", "cliente", "segmento", "canal", "categoria", "producto", "subtotal", "margen_bruto", "estado"]
faltan = [c for c in esperadas if c not in libro.columns]
revision = [
    ("Están todas las columnas que usa el informe", not faltan, f"Faltan: {', '.join(faltan)}" if faltan else ""),
    ("Las fechas se reconocen como fechas", libro["fecha"].notna().all(), f"{libro['fecha'].isna().sum()} fechas no válidas"),
    ("subtotal es numérico", pd.api.types.is_numeric_dtype(libro["subtotal"]), "Hay textos en subtotal (clave 2)"),
    ("No hay subtotales vacíos", libro["subtotal"].notna().all(), f"{libro['subtotal'].isna().sum()} vacíos"),
]
for texto, ok, detalle in revision:
    display(Markdown(f"{'✓' if ok else '✗'} {texto}" + ("" if ok else f" — **{detalle}**")))
anuladas = (libro["estado"] == "Anulada").sum()
display(Markdown(f"ℹ️ El libro trae **{anuladas} líneas anuladas**. El informe las excluye porque no son venta."))
datos_ok = libro[libro["estado"] != "Anulada"].copy()

## Paso 5 · Configura el informe
Elige y ejecuta. Luego ejecuta los cinco gráficos (o usa *Entorno de ejecución › Ejecutar celdas posteriores*).

In [ ]:
# @title Configuración del informe
anio = "2025"  # @param ["2025", "2024", "Todos"]
sede = "Todas"  # @param ["Todas", "Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga", "Pereira"]
medida = "Venta neta"  # @param ["Venta neta", "Margen bruto"]
sel = datos_ok if anio == "Todos" else datos_ok[datos_ok.anio == int(anio)]
if sede != "Todas": sel = sel[sel.ciudad == sede]
col = "subtotal" if medida == "Venta neta" else "margen_bruto"
alcance = f"{'toda la empresa' if sede == 'Todas' else sede} · {'2024–2025' if anio == 'Todos' else anio}"
informe = []   # aquí se guardan los gráficos para exportarlos
display(Markdown(f"Informe de **{medida.lower()}** para **{alcance}**: {miles(len(sel))} líneas."))

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿En qué meses y dónde se concentra la venta? |
> | **Qué se grafica** | Filas: sedes (o líneas, si elegiste una sede) · Columnas: meses · Color: medida elegida |
> | **Por qué esta forma** | Dos categorías cruzadas: el color muestra el patrón de 72 cifras de un vistazo. |
> | **Qué se descartó** | Una línea por sede: seis líneas entrecruzadas. |
> | **Cómo se lee** | Las celdas más oscuras son los momentos fuertes. |
> | **En Excel 365** | Tabla dinámica + Formato condicional › Escalas de color. Requiere armar la tabla primero. |

In [ ]:
# @title Gráfico 1 · Mapa de calor
filas = "ciudad" if sede == "Todas" else "categoria"
p = sel.pivot_table(index=filas, columns="mes", values=col, aggfunc="sum").fillna(0) / 1e6
p = p.loc[p.sum(axis=1).sort_values(ascending=False).index]
i, j = np.unravel_index(np.argmax(p.values), p.shape)
fig = px.imshow(p, x=[MESES[m - 1] for m in p.columns], text_auto=".0f", aspect="auto", color_continuous_scale=["#EEF2FF", AZUL, "#00006A"],
                title=f"{p.index[i]} en {MESES[p.columns[j] - 1].lower()} es el punto más alto ({alcance}, millones)")
fig.update_layout(coloraxis_colorbar_title="M$", xaxis_title="", yaxis_title="", height=400)
fig.show(); informe.append(fig)

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué tan concentrada está la venta en pocos clientes? |
> | **Qué se grafica** | Barras: aporte de cada 10 % de clientes · Línea: acumulado · Línea punteada: 80 % |
> | **Por qué esta forma** | El Pareto responde si unos pocos explican la mayoría; un solo eje de porcentaje evita la trampa de dos ejes. |
> | **Qué se descartó** | Barras de cientos de clientes: ilegibles. |
> | **Cómo se lee** | Mira en qué grupo la línea cruza el 80 %. |
> | **En Excel 365** | Insertar › Gráfico estadístico › Pareto, sobre una tabla de clientes ordenada. |

In [ ]:
# @title Gráfico 2 · Pareto de clientes
c = sel.groupby("cliente")[col].sum().sort_values(ascending=False); n = len(c)
dec = pd.Series([c.iloc[int(n * d / 10):int(n * (d + 1) / 10)].sum() / c.sum() for d in range(10)], index=[f"{(d + 1) * 10} %" for d in range(10)])
fig = go.Figure([go.Bar(x=dec.index, y=dec.values, name="Aporte del grupo", marker_color=AZUL),
                 go.Scatter(x=dec.index, y=dec.cumsum(), name="Acumulado", mode="lines+markers", line=dict(color=MAGENTA))])
fig.add_hline(y=0.8, line_dash="dot", line_color=MAGENTA)
fig.update_layout(title=f"El 20 % de los clientes explica el {porc(dec.iloc[:2].sum())} ({alcance})", yaxis_tickformat=".0%",
                  xaxis_title=f"{n} clientes, del que más compra al que menos", legend_title="")
fig.show(); informe.append(fig)

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué líneas y productos pesan más? |
> | **Qué se grafica** | Rectángulos: líneas de producto y, dentro, sus productos; el área es la medida |
> | **Por qué esta forma** | Muestra composición en dos niveles a la vez; se puede hacer clic para entrar en una línea. |
> | **Qué se descartó** | Un anillo con 73 productos: imposible de leer. |
> | **Cómo se lee** | Los rectángulos grandes son los que más pesan. Haz clic en una línea para ver sus productos. |
> | **En Excel 365** | Insertar › Gráfico de rectángulos (Treemap). |

In [ ]:
# @title Gráfico 3 · Mapa de árbol línea → producto
t = sel.groupby(["categoria", "producto"], as_index=False)[col].sum()
t = t[t[col] > 0]
fig = px.treemap(t, path=[px.Constant("Total"), "categoria", "producto"], values=col, color="categoria",
                 color_discrete_sequence=[AZUL, TEAL, MAGENTA, "#7A5AC8", "#3E8E5E"])
top = t.groupby("categoria")[col].sum().idxmax()
fig.update_layout(title=f"{top} es la línea que más pesa en {medida.lower()} ({alcance})", height=480, margin=dict(t=70, l=5, r=5, b=5))
fig.update_traces(hovertemplate="%{label}<br>%{value:,.0f}<extra></extra>")
fig.show(); informe.append(fig)

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cómo evolucionó cada sede mes a mes? |
> | **Qué se grafica** | Una pequeña gráfica de líneas por sede (o por línea), con el mismo eje |
> | **Por qué esta forma** | Varias gráficas pequeñas evitan una maraña de seis líneas y permiten comparar formas. |
> | **Qué se descartó** | Seis líneas en un solo gráfico: se cruzan y nadie las sigue. |
> | **Cómo se lee** | Compara la forma de cada curva: quién crece, quién cae, quién es estacional. |
> | **En Excel 365** | En Excel exige seis gráficos alineados a mano. |

In [ ]:
# @title Gráfico 4 · Una gráfica por sede
grupo = "ciudad" if sede == "Todas" else "categoria"
s = sel.assign(periodo=sel.fecha.dt.to_period("M").dt.to_timestamp()).groupby([grupo, "periodo"], as_index=False)[col].sum()
s["millones"] = s[col] / 1e6
orden = s.groupby(grupo)["millones"].sum().sort_values(ascending=False).index.tolist()
fig = px.line(s, x="periodo", y="millones", facet_col=grupo, facet_col_wrap=3, color_discrete_sequence=[AZUL], category_orders={grupo: orden})
eje_meses(fig, s["periodo"], cada=3)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(title=f"{medida} mensual por {'sede' if grupo == 'ciudad' else 'línea'} ({alcance}, millones)", height=520, showlegend=False)
fig.update_yaxes(title=""); fig.update_xaxes(title="")
fig.show(); informe.append(fig)

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Los vendedores que más venden dejan más margen? |
> | **Qué se grafica** | Un punto por vendedor · Eje X: venta · Eje Y: margen % |
> | **Por qué esta forma** | Relación entre dos medidas, un punto por persona. |
> | **Qué se descartó** | Barras de cada medida por separado: no se ve la relación. |
> | **Cómo se lee** | Los puntos abajo a la derecha venden mucho con poco margen: ¿descuentos? |
> | **En Excel 365** | Insertar › Dispersión (X, Y). |

In [ ]:
# @title Gráfico 5 · Dispersión de vendedores
g = sel.groupby("vendedor").agg(venta=("subtotal", "sum"), margen=("margen_bruto", "sum"))
g = g[g.venta > 0]; g["margen %"] = g.margen / g.venta
fig = px.scatter(g.reset_index(), x="venta", y="margen %", hover_name="vendedor", color_discrete_sequence=[AZUL])
fig.update_traces(marker_size=13, marker_line_color="white", marker_line_width=1.5)
fig.update_layout(title=f"Venta y margen por vendedor ({alcance})", yaxis_tickformat=".0%", xaxis_title="Venta neta (pesos)")
fig.show(); informe.append(fig)

## Paso 6 · Exportar el informe
Crea una página web con los cinco gráficos. Funciona sin Colab y sin internet: se puede enviar por correo o Teams.

In [ ]:
# @title Crear y descargar informe_interactivo_Andina.html
from datetime import date
partes = [f"""<html><head><meta charset="utf-8"><title>Informe interactivo · Distribuidora Andina</title>
<style>body{{font-family:Segoe UI,Arial,sans-serif;max-width:1100px;margin:24px auto;padding:0 16px;color:#11153A}}
h1{{color:#00006A;margin-bottom:4px}} .meta{{color:#566080;margin-bottom:24px}} .g{{margin:28px 0;border-top:1px solid #D6DCEA;padding-top:12px}}</style></head><body>
<h1>Distribuidora Andina S.A.S. · Informe interactivo</h1>
<div class="meta">{medida} · {alcance} · sin facturas anuladas · generado el {date.today():%Y-%m-%d} con Colab · Fuente: {archivo.split('/')[-1]}</div>"""]
for k, f in enumerate(informe):
    partes.append('<div class="g">' + f.to_html(full_html=False, include_plotlyjs=(k == 0)) + "</div>")
partes.append('<p class="meta">Pasa el mouse sobre los gráficos para ver los valores. Datos ficticios · Excel Corporativo · 12 Claves · Dr. Bits.</p></body></html>')
with open("informe_interactivo_Andina.html", "w", encoding="utf-8") as fh:
    fh.write("".join(partes))

if EN_COLAB:
    from google.colab import files
    files.download("informe_interactivo_Andina.html")
    display(Markdown("Descarga iniciada: **informe_interactivo_Andina.html**. Si no aparece, búscalo en el panel de archivos (ícono de carpeta) › clic derecho › Descargar."))
else:
    display(Markdown("Archivo creado: **informe_interactivo_Andina.html** (en la carpeta del cuaderno)."))

## Resumen
* Excel guarda y organiza los datos; Colab los revisa y los convierte en análisis.
* Revisar antes de graficar evita conclusiones sobre datos equivocados.
* Un HTML interactivo se comparte sin que el lector instale nada.

**Siguiente:** [Taller 2 · De Colab a Excel](https://omarli-360.github.io/excel-corporativo-12-claves/#taller-2)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>